# 08. 软件安装与压缩包

AGDR VLA 入门教程 · 第 2 章 · 第 2 周

- **学习目标**：会用 `apt` 管理软件，会用 `curl` / `wget` 下载，会用 `tar` 打包和解压
- **前置知识**：[05-用户与权限](./05-用户与权限.ipynb)
- **预计学习时长**：1 小时

## apt：Ubuntu 的软件管理器

| 命令 | 作用 |
| --- | --- |
| `sudo apt update` | 更新**软件列表**，不安装任何东西 |
| `sudo apt upgrade` | 把已安装的软件升级到新版本 |
| `sudo apt install <包>` | 安装 |
| `sudo apt remove <包>` | 卸载 |
| `apt search <关键词>` | 搜索 |
| `apt show <包>` | 查看详情 |
| `apt list --installed` | 列出已安装的包 |

**`update` 和 `upgrade` 不一样**，这是最容易混淆的一点。一般安装软件前先 `update`，让系统知道最新的软件列表。

> 这里的命令要用 `sudo`，需要在终端里运行。其他发行版的软件管理器不同，例如 Fedora 用 `dnf`。

请在**终端**里安装后面要用的工具：

```bash
sudo apt update
sudo apt install curl wget htop tree zip unzip tmux rsync
```

看到 `Do you want to continue? [Y/n]` 时，读一下要安装的内容，再按回车确认。

安装完成后，运行下面的单元格检查。

In [ ]:
%%bash
for tool in curl wget htop tree zip unzip tmux rsync; do
    if command -v "$tool" > /dev/null; then
        echo "已安装: $tool"
    else
        echo "未安装: $tool"
    fi
done

### 报错了怎么办

| 报错 | 常见原因 | 处理 |
| --- | --- | --- |
| `Unable to locate package` | 包名错了，或者软件列表太旧 | 先 `sudo apt update`，再用 `apt search` 确认 |
| `Could not get lock` | 另一个 `apt` 或系统自动更新正在运行 | 等它结束，**不要手动删除锁文件** |
| `Permission denied` | 忘了 `sudo` | 加上 `sudo` |
| 下载很慢或连接失败 | 网络问题 | 在 Ubuntu 的「软件和更新」中切换国内镜像源 |

## 别的安装方式

并不是所有软件都在 `apt` 里：

| 方式 | 例子 | 说明 |
| --- | --- | --- |
| `.deb` 包 | `sudo apt install ./xxx.deb` | 路径里的 `./` 不能少，否则会当成包名去搜 |
| 官方脚本 | 官网文档给出的 `curl ... \| sh` | 见下面的警告 |
| 语言包管理器 | `pip`、`conda` | 第 4 章讲；Python 库不要用 `sudo pip` |
| 从源码编译 | `make`、`cmake` | 第 6 章前后会遇到 |

> ⚠️ `curl ... | bash` 相当于把网上下载的内容直接交给 Shell 运行。只对**可信的官方来源**这样做，最好先下载下来读一读。不要运行来路不明的脚本，特别是还带 `sudo` 的。

## 下载文件：curl 和 wget

| 目的 | 命令 |
| --- | --- |
| 下载，文件名沿用网址里的 | `wget <网址>` |
| 下载，并指定文件名 | `wget -O model.pt <网址>` 或 `curl -L -o model.pt <网址>` |
| 断点续传（大文件） | `wget -c <网址>` 或 `curl -L -C - -O <网址>` |
| 把内容输出到屏幕 | `curl -sL <网址>` |

`curl` 的 `-L` 表示跟随跳转，下载 GitHub 文件时几乎都需要。

请在终端里试试（要联网）：

```bash
curl -sI https://github.com | head -n 1
```

`-I` 表示只请求响应头。看到 `200` 说明网络通畅。

下载大文件后，可以用校验值确认文件完整：

In [ ]:
%%bash
rm -rf /tmp/linux-demo && mkdir -p /tmp/linux-demo && cd /tmp/linux-demo
echo "pretend this is a model" > model.pt
sha256sum model.pt

把结果和发布页面上给出的校验值对比，一致就说明文件完好。

## tar：打包与解压

数据集和发布包经常是 `.tar.gz`。

| 目的 | 命令 |
| --- | --- |
| 打包并压缩 | `tar -czf 包名.tar.gz 目录/` |
| 查看内容（不解压） | `tar -tzf 包名.tar.gz` |
| 解压到当前目录 | `tar -xzf 包名.tar.gz` |
| 解压到指定目录 | `tar -xzf 包名.tar.gz -C 目标目录/` |

字母的含义：`c` 创建，`x` 解压，`t` 查看，`z` 用 gzip，`f` 后面跟文件名（所以 `f` 要放在最后）。

In [ ]:
%%bash
cd /tmp/linux-demo
mkdir -p dataset/episodes
echo "data1" > dataset/episodes/1.txt
echo "data2" > dataset/episodes/2.txt
tar -czf dataset.tar.gz dataset/
ls -lh dataset.tar.gz
echo "--- 查看内容"
tar -tzf dataset.tar.gz

In [ ]:
%%bash
cd /tmp/linux-demo
mkdir restored
tar -xzf dataset.tar.gz -C restored/
find restored -type f

> 解压别人的压缩包前，先用 `tar -tzf` 看看里面的结构，防止文件散落一地，或者覆盖同名文件。

**不同的压缩格式**

| 后缀 | 解压命令 |
| --- | --- |
| `.tar.gz`、`.tgz` | `tar -xzf 文件` |
| `.tar.xz` | `tar -xJf 文件` |
| `.tar` | `tar -xf 文件` |
| `.zip` | `unzip 文件 -d 目录` |
| `.gz`（单个文件） | `gunzip 文件` |

现代 `tar` 解压时能自动识别压缩方式，所以 `tar -xf` 通常就够了。

## 动手试试

在终端里完成，不计分：

1. 用 `apt show` 查看 `tmux` 的版本和说明
2. 把 `/tmp/linux-demo/dataset` 打包成 `/tmp/backup.tar.gz`
3. 不解压，确认包里有 `1.txt`
4. 解压到 `/tmp/unpacked`（目录要提前创建）

<details>
<summary>参考答案</summary>

```bash
apt show tmux
cd /tmp/linux-demo
tar -czf /tmp/backup.tar.gz dataset/
tar -tzf /tmp/backup.tar.gz | grep 1.txt
mkdir -p /tmp/unpacked
tar -xzf /tmp/backup.tar.gz -C /tmp/unpacked
```

</details>

## 小结

| 你应该能 | 相关内容 |
| --- | --- |
| 区分 `apt update` 和 `apt upgrade`，并安装软件 | apt |
| 判断运行安装脚本的风险 | 别的安装方式 |
| 用 `curl`、`wget` 下载文件 | 下载文件 |
| 用 `tar` 打包、查看、解压 | tar |

下一节：[09-远程连接](./09-远程连接.ipynb)